# Proxy Clinical: Block 2 pilot launcher (Colab Files, no Drive)

Thin launcher. No training logic lives here; everything runs from the repo.
Runtime: **A100 (high-RAM)**.

**Upload first** (Files panel on the left, upload icon): `proxy-clinical-block2.tar.gz`. It contains the
repo and the pilot JSONL, so it is the only file needed. Optional: add `HF_TOKEN` in the Secrets panel
(only needed for gated bases such as Llama).

`/content` is wiped when the runtime is deleted, not when the session is restarted. So: run cell 1, cell 2,
then **Runtime > Restart session**, then cells 3, 4, 5. The uploaded tarball and the extracted repo survive
the restart. Cell 5 zips the run folder and downloads it; do that before the runtime times out, because
nothing here persists otherwise.


In [ ]:
# 1. Extract the uploaded tarball into /content/proxy-clinical
import os, subprocess, glob
TARBALLS = sorted(glob.glob('/content/proxy-clinical-block*.tar.gz'))
assert TARBALLS, 'upload proxy-clinical-block2.tar.gz to /content via the Files panel first'
TARBALL = TARBALLS[-1]
REPO = '/content/proxy-clinical'
if not os.path.isdir(REPO):
    subprocess.run(['tar', '-xzf', TARBALL, '-C', '/content'], check=True)
print('repo:', REPO)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
for name in ('corpus.jsonl', 'train.jsonl', 'val.jsonl', 'meta.json'):
    p = f'{REPO}/data/pilot/{name}'
    print(f'{name:14s}', 'ok' if os.path.exists(p) else 'MISSING', os.path.getsize(p) if os.path.exists(p) else '')


In [ ]:
# 2. Pinned environment (replaces Colab's preinstalled stack). Then: Runtime > Restart session, continue at cell 3.
# Colab preinstalls torchvision/torchaudio (built against its own torch) and an old torchao; after torch is replaced
# they fail to load, and transformers/peft surface that as bogus import errors. Nothing here needs them.
%pip uninstall -y -q torchvision torchaudio torchao bitsandbytes
%pip install -q -r /content/proxy-clinical/requirements-colab.txt
import subprocess, sys
out = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'],
                     capture_output=True, text=True).stdout.strip()
print('torch after install:', out)
if out.endswith('False'):
    # Driver older than the CUDA 13 wheel needs: same torch version, CUDA 12.6 build.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'torch==2.14.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
    print('reinstalled torch (cu126 build)')
chk = subprocess.run([sys.executable, '-c', 'import transformers, trl, peft; from transformers import AutoModelForCausalLM; print("imports ok")'],
                     capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-800:])
print('Now: Runtime > Restart session, then run cells 3, 4, 5.')


In [ ]:
# 3. (after restart) Sanity checks + preflight: import probes, then a one-step CPU training on the tiny stand-in.
# Fails in ~1 minute if the environment is broken, BEFORE the 6 GB model download. Also pulls HF_TOKEN from Secrets.
import os, hashlib
os.chdir('/content/proxy-clinical')
for name in ('corpus.jsonl', 'train.jsonl', 'val.jsonl'):
    p = f'data/pilot/{name}'
    print(f'{name:14s}', hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16], os.path.getsize(p), 'bytes')
# Expected for the cafebabe pilot: corpus fe0dad56f62d7bc5, train 0ae148bc22dd8445, val c75668ed7faddffd
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Secrets')
except Exception as e:
    print('HF_TOKEN not set (fine for Qwen2.5, required for Llama):', type(e).__name__)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!bash scripts/preflight.sh


In [ ]:
# 4. Run the pilot: train -> infer -> evaluate -> determinism. Re-run with the same RUN_ID to resume within this session.
import datetime, os
os.chdir('/content/proxy-clinical')
RUN_ID = os.environ.get('RUN_ID') or f"pilot-qwen2.5-3b-{datetime.date.today():%Y%m%d}"
OUT = f'/content/runs/{RUN_ID}'
print('run folder:', OUT)
!bash scripts/run_pilot.sh configs/pilot.yaml "$OUT"


In [ ]:
# 5. Package the run folder (adapter, manifests, predictions, eval, determinism, pip freeze; checkpoints excluded) and download it
import os, subprocess, datetime
from google.colab import files
os.chdir('/content')
RUN_ID = os.environ.get('RUN_ID') or f"pilot-qwen2.5-3b-{datetime.date.today():%Y%m%d}"
zip_path = f'/content/{RUN_ID}.zip'
subprocess.run(['bash', '-c', f"cd /content/runs && zip -qr {zip_path} {RUN_ID} -x '{RUN_ID}/checkpoint-*/*'"], check=True)
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 1), 'MB')
print(open(f'/content/runs/{RUN_ID}/eval_report.md').read())
files.download(zip_path)


The run folder holds `adapter/` (LoRA weights + tokenizer), `run_manifest.json`, `config.yaml`,
`predictions.jsonl` (+ manifest), `eval_report.md`, `eval.json`, `determinism.json`, `pip_freeze.txt`.

The determinism claim proven above is: same checkpoint, same inputs, same pinned environment, same session,
byte-identical output. It is not a cross-GPU claim.
